# Mole Classifier:
This notebook trains a simple image classifier using Keras to tell the difference between benign moles and melanoma.


## Step 1: Import Libraries

In [1]:
import tensorflow as tf
from tensorflow import keras
import matplotlib.pyplot as plt

## Step 2: Load the Images

Keras can read images directly from folders. Each subfolder name becomes the label ("benign" or "malignant").

We resize all images to 64×64 pixels so they are all the same size.

In [3]:
IMAGE_SIZE = (64, 64)   # resize all images to 64x64 pixels
BATCH_SIZE = 16         # process 16 images at a time

def load_data(folder_path):
    dataset = keras.utils.image_dataset_from_directory(
        folder_path,
        image_size=IMAGE_SIZE,
        batch_size=BATCH_SIZE,
        label_mode="binary"  # two classes: 0 or 1
    )
    return dataset

train_data = load_data("melanoma_cancer_dataset/train")
test_data  = load_data("melanoma_cancer_dataset/test")

# Print the class names Keras found
print("Classes:", train_data.class_names)

Found 9605 files belonging to 2 classes.
Found 1000 files belonging to 2 classes.
Classes: ['benign', 'malignant']


## Step 3: Preview Some Images

Let's look at a few images so we know the data loaded correctly.

In [ ]:
def show_sample_images(dataset):
    class_names = dataset.class_names
    images, labels = next(iter(dataset))  # grab the first batch

    plt.figure(figsize=(10, 4))
    for i in range(6):
        plt.subplot(2, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        label_index = int(labels[i].numpy())
        plt.title(class_names[label_index])
        plt.axis("off")
    plt.tight_layout()
    plt.show()

show_sample_images(train_data)

## Step 4: Normalize the Images

Pixel values go from 0–255. We scale them to 0–1 so the model trains more smoothly.

In [ ]:
def normalize(image, label):
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

train_data = train_data.map(normalize)
test_data  = test_data.map(normalize)

## Step 5: Build the Model

Our neural network has three parts:
1. **Convolutional layers** — detect shapes and textures in the image
2. **Flatten** — turn the 2D feature map into a 1D list
3. **Dense layers** — make the final benign/melanoma decision

In [ ]:
def build_model():
    model = keras.Sequential([
        # --- Feature detection ---
        keras.layers.Conv2D(16, (3, 3), activation="relu", input_shape=(64, 64, 3)),
        keras.layers.MaxPooling2D(),

        keras.layers.Conv2D(32, (3, 3), activation="relu"),
        keras.layers.MaxPooling2D(),

        # --- Decision making ---
        keras.layers.Flatten(),
        keras.layers.Dense(64, activation="relu"),
        keras.layers.Dense(1, activation="sigmoid")  # output: 0 = benign, 1 = malignant
    ])

    model.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    return model

model = build_model()
model.summary()

EPOCHS = 5

history = model.fit(
    train_data,
    epochs=EPOCHS,
    validation_data=test_data
)

## Step 6: Train the Model

We train for **5 epochs** (5 passes through the training data). This keeps training fast while still showing the model learning.

In [ ]:
EPOCHS = 5

history = model.fit(
    train_data,
    epochs=EPOCHS,
    validation_data=test_data
)

## Step 7: Plot Training Results

Let's see how accuracy and loss changed over each epoch.

In [ ]:
def plot_training(history):
    epochs = range(1, len(history.history["accuracy"]) + 1)

    plt.figure(figsize=(12, 4))

    # Accuracy plot
    plt.subplot(1, 2, 1)
    plt.plot(epochs, history.history["accuracy"],     label="Train Accuracy")
    plt.plot(epochs, history.history["val_accuracy"], label="Test Accuracy")
    plt.title("Accuracy Over Time")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()

    # Loss plot
    plt.subplot(1, 2, 2)
    plt.plot(epochs, history.history["loss"],     label="Train Loss")
    plt.plot(epochs, history.history["val_loss"], label="Test Loss")
    plt.title("Loss Over Time")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()

    plt.tight_layout()
    plt.show()

plot_training(history)

## Step 8: Evaluate on the Test Set

In [ ]:
def evaluate_model(model, test_data):
    loss, accuracy = model.evaluate(test_data)
    print(f"Test Loss:     {loss:.4f}")
    print(f"Test Accuracy: {accuracy:.2%}")

evaluate_model(model, test_data)

## Step 9: Predict on Individual Images

Let's look at a few test images and see what the model predicts.

In [ ]:
def show_predictions(model, dataset, num_images=6):
    class_names = dataset.class_names
    images, true_labels = next(iter(dataset))  # grab one batch

    predictions = model.predict(images)

    plt.figure(figsize=(12, 5))
    for i in range(num_images):
        plt.subplot(2, 3, i + 1)
        plt.imshow(images[i].numpy())

        predicted_index = int(predictions[i][0] > 0.5)  # 0 or 1
        true_index      = int(true_labels[i].numpy())

        predicted_label = class_names[predicted_index]
        true_label      = class_names[true_index]

        color = "green" if predicted_index == true_index else "red"
        plt.title(f"Pred: {predicted_label}\nTrue: {true_label}", color=color)
        plt.axis("off")

    plt.tight_layout()
    plt.show()

show_predictions(model, test_data)